# ByteMoE feasibility gates — Colab runner

Run this notebook on a **T4 GPU** (`Runtime` → `Change runtime type` → `T4 GPU`). It runs the three early go/no-go checks:

1. **E0:** all packed SwiGLU blocks reproduce the original expert and model next-token prediction.
2. **E1:** 25%/50% importance-ordered prefixes beat equally sized random prefixes.
3. **E2:** candidate blocks retain a transfer/kernel advantage over whole-expert loading.

The notebook mounts Google Drive, then clones [sahilaf/ByteMoE](https://github.com/sahilaf/ByteMoE) into `MyDrive/ByteMoEColab/repo`. Model downloads and result files are also stored in Drive, so a reconnect resumes from the last completed gate.

In [ ]:
# Confirm that Colab assigned a GPU. Stop here if this prints False.
import torch
assert torch.cuda.is_available(), 'Enable a T4 GPU: Runtime > Change runtime type > T4 GPU.'
print(torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

In [ ]:
# Persistent setup: repository, results, and Hugging Face cache all live in Google Drive.
# On a new Colab runtime, run this cell before any model-loading cell.
from google.colab import drive
from pathlib import Path
import os
import subprocess

drive.mount('/content/drive', force_remount=False)
PERSIST_ROOT = Path('/content/drive/MyDrive/ByteMoEColab')
PROJECT_DIR = PERSIST_ROOT / 'repo'
HF_HOME = PERSIST_ROOT / 'hf_cache'
RESULTS_DIR = PROJECT_DIR / 'results'
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(exist_ok=True)
# Hugging Face resumes incomplete shard downloads from this persistent cache.
os.environ['HF_HOME'] = str(HF_HOME)
os.environ['HF_HUB_CACHE'] = str(HF_HOME / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(HF_HOME / 'hub')

REPO_URL = 'https://github.com/sahilaf/ByteMoE.git'
if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)

assert (PROJECT_DIR / 'requirements.txt').exists(), f'Missing requirements.txt in {PROJECT_DIR}'
RESULTS_DIR.mkdir(exist_ok=True)
%cd {PROJECT_DIR}
print('Project:', PROJECT_DIR)
print('Persistent results:', RESULTS_DIR)
print('Persistent model cache:', HF_HOME)
!git rev-parse --short HEAD

In [ ]:
# Install the runner dependencies. The OLMoE checkpoint downloads on the first model command.
!pip -q install -r requirements.txt
!python -m pytest -q

## Model configuration

The default model is the primary OLMoE target. Change `MODEL_ID` only if you are deliberately testing another compatible MoE model. Use `fp16` on a T4.

In [ ]:
MODEL_ID = 'allenai/OLMoE-1B-7B-0924'
DTYPE = 'fp16'
EXPERT_INDEX = 0  # Update after the next cell if needed.
BLOCKS = 16
FORCE_RERUN = False  # Change to True only when intentionally replacing a completed result.

E0_OUTPUT = RESULTS_DIR / 'e0_exactness.json'
E1_OUTPUT = RESULTS_DIR / 'e1_prefix_viability.json'
E2_OUTPUT = RESULTS_DIR / 'e2_microbenchmark.csv'

def run_or_resume(name, command, output_path):
    if output_path.exists() and not FORCE_RERUN:
        print(f'{name} already completed — reusing {output_path}. Set FORCE_RERUN=True to run it again.')
        return
    print(f'Running {name}; results will persist in Google Drive.')
    subprocess.run(command, cwd=PROJECT_DIR, env=os.environ.copy(), check=True)

# Optional: authenticate only if Hugging Face asks for it.
# !huggingface-cli login

In [ ]:
# This prints [index] expert-path: hidden=... intermediate=... .
# Select an index that is likely to be routed by the supplied prompts (0 is a sensible first try).
!python -m scripts.e0_exactness --model {MODEL_ID} --dtype {DTYPE} --list-experts

## E0 — exactness

This must print `passed: True`. If it does not, **stop**: packing or the model adapter needs correction before approximate prefixes have meaning.

In [ ]:
run_or_resume(
    'E0',
    ['python', '-m', 'scripts.e0_exactness', '--model', MODEL_ID, '--dtype', DTYPE,
     '--expert-index', str(EXPERT_INDEX), '--blocks', str(BLOCKS), '--prompt-copies', '32'],
    E0_OUTPUT,
)
print(E0_OUTPUT.read_text())

## E1 — prefix viability

This starts with a norm-based importance heuristic. It is a quick feasibility test, not the final learned predictor. Continue only when `passed: True` and the ordered prefix beats random by a material margin at 25% or 50%.

In [ ]:
run_or_resume(
    'E1',
    ['python', '-m', 'scripts.e1_prefix_viability', '--model', MODEL_ID, '--dtype', DTYPE,
     '--expert-index', str(EXPERT_INDEX), '--fractions', '0.25', '0.5', '--prompt-copies', '32'],
    E1_OUTPUT,
)
print(E1_OUTPUT.read_text())

## E2 — transfer/kernel microbenchmark

Copy `hidden` and `intermediate` from the expert-list output into the next cell. E2 tests pinned-memory H2D copies and SwiGLU compute at several widths plus the whole-expert reference.

In [ ]:
# Replace these with dimensions printed by the expert list.
HIDDEN_SIZE = 2048
INTERMEDIATE_SIZE = 8192
BLOCK_WIDTHS = '64 128 256 512 1024 2048'

run_or_resume(
    'E2',
    ['python', '-m', 'scripts.e2_microbenchmark', '--hidden-size', str(HIDDEN_SIZE),
     '--intermediate-size', str(INTERMEDIATE_SIZE), '--block-widths', *BLOCK_WIDTHS.split(), '--repetitions', '100'],
    E2_OUTPUT,
)
print(E2_OUTPUT.read_text())

## Decision

Continue only if all three conditions hold:

- E0: `passed: True` and top-1 agreement is 1.0.
- E1: `passed: True`; the importance prefix has at least the configured 0.05 agreement lift above random.
- E2: `aggregate/full` is below 1.0 for the block width and prefix fraction that E1 found useful.

All reports persist in `MyDrive/ByteMoEColab/repo/results/`. After a disconnect, re-enable the T4, run the setup and dependency cells, then run only the next unfinished gate.

In [ ]:
# Optional: download copies. They are already safely persisted in Google Drive.
from google.colab import files
files.download(str(E0_OUTPUT))
files.download(str(E1_OUTPUT))
files.download(str(E2_OUTPUT))